In [1]:
"""
Mule Account Detection using LightGBM + CatBoost Ensemble

Objective:
----------
This pipeline detects suspicious mule accounts from banking feature data.
The target column is F3924:
    0 = Normal Account
    1 = Suspicious / Mule Account

The final output is not only a binary prediction, but also a risk score from 0 to 100.

Why this approach?
------------------
The dataset is highly imbalanced:
    Normal accounts: 9001
    Mule accounts: 81

Because of this imbalance, accuracy is not a reliable metric. A model could predict
all accounts as normal and still get very high accuracy. Therefore, this pipeline
focuses on:
    - PR-AUC
    - Recall
    - F1-score
    - Confusion Matrix

Pipeline Steps:
---------------
1. Load dataset
2. Remove unnecessary / empty / suspicious columns
3. Separate target variable F3924 from input features
4. Apply feature engineering:
    - Convert date column into numerical age feature
    - Extract numerical duration and type from coded duration fields
    - Create missing-value indicator columns
5. Remove noisy features:
    - Columns with more than 95% missing values
    - Constant columns
6. Convert categorical columns into category format
7. Handle class imbalance using scale_pos_weight
8. Train LightGBM using Stratified 5-Fold Cross Validation
9. Train CatBoost using the same folds
10. Generate out-of-fold predictions from both models
11. Create weighted ensemble of LightGBM and CatBoost predictions
12. Optimize threshold using F1-score
13. Generate final predictions and risk scores
14. Save output files:
    - optimized_mule_account_risk_scores.csv
    - optimized_lgb_feature_importance.csv

Model Details:
--------------
LightGBM:
    Used because it handles high-dimensional tabular data, missing values,
    class imbalance, and feature importance efficiently.

CatBoost:
    Used because it handles categorical features well and can capture patterns
    different from LightGBM.

Ensemble:
    The final probability is calculated as:

        final_probability =
            weight * LightGBM_probability
            + (1 - weight) * CatBoost_probability

    The best weight is selected based on PR-AUC.

Risk Score:
-----------
The final probability is converted into a risk score:

    risk_score = probability * 100

Example:
    probability = 0.91
    risk_score = 91

This helps banks prioritize accounts for investigation.

Output Interpretation:
----------------------
The output CSV contains:
    actual_label:
        True label from dataset

    lgb_probability:
        Mule probability predicted by LightGBM

    cat_probability:
        Mule probability predicted by CatBoost

    ensemble_probability:
        Final combined probability

    risk_score_0_100:
        Risk score scaled from 0 to 100

    predicted_label:
        Final predicted class after threshold optimization

Evaluation:
-----------
The model is evaluated using:
    ROC-AUC:
        Measures overall ranking performance.

    PR-AUC:
        More important for imbalanced fraud detection.

    Precision:
        Out of all accounts flagged as mule, how many were actually mule.

    Recall:
        Out of all actual mule accounts, how many were detected.

    F1-score:
        Balance between precision and recall.

    Confusion Matrix:
        Shows true positives, false positives, true negatives, and false negatives.

Note:
-----
Since this is a fraud detection problem, recall is especially important because
missing a mule account can be more costly than investigating extra false alerts.
"""

'\nMule Account Detection using LightGBM + CatBoost Ensemble\n\nObjective:\n----------\nThis pipeline detects suspicious mule accounts from banking feature data.\nThe target column is F3924:\n    0 = Normal Account\n    1 = Suspicious / Mule Account\n\nThe final output is not only a binary prediction, but also a risk score from 0 to 100.\n\nWhy this approach?\n------------------\nThe dataset is highly imbalanced:\n    Normal accounts: 9001\n    Mule accounts: 81\n\nBecause of this imbalance, accuracy is not a reliable metric. A model could predict\nall accounts as normal and still get very high accuracy. Therefore, this pipeline\nfocuses on:\n    - PR-AUC\n    - Recall\n    - F1-score\n    - Confusion Matrix\n\nPipeline Steps:\n---------------\n1. Load dataset\n2. Remove unnecessary / empty / suspicious columns\n3. Separate target variable F3924 from input features\n4. Apply feature engineering:\n    - Convert date column into numerical age feature\n    - Extract numerical duration and

In [ ]:


import pandas as pd
import numpy as np
import lightgbm as lgb

from catboost import CatBoostClassifier

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    classification_report,
    confusion_matrix,
    precision_recall_curve
)

# =========================
# 1. Load Data
# =========================

DATA_PATH = "DataSet.csv"
TARGET = "F3924"

df = pd.read_csv(DATA_PATH)

print("Original shape:", df.shape)
print("Target distribution:")
print(df[TARGET].value_counts())

# =========================
# 2. Cleaning
# =========================

drop_cols = []

if "Unnamed: 0" in df.columns:
    drop_cols.append("Unnamed: 0")

# possible leakage / ID-like suspicious column
if "F2230" in df.columns:
    drop_cols.append("F2230")

empty_cols = df.columns[df.isna().mean() == 1].tolist()
drop_cols += empty_cols

df = df.drop(columns=list(set(drop_cols)), errors="ignore")

y = df[TARGET].astype(int)
X = df.drop(columns=[TARGET])

# =========================
# 3. Feature Engineering
# =========================

if "F3888" in X.columns:
    date_col = pd.to_datetime(X["F3888"], errors="coerce")
    max_date = date_col.max()
    X["F3888_days_old"] = (max_date - date_col).dt.days
    X = X.drop(columns=["F3888"])

if "F3889" in X.columns:
    X["F3889_days"] = X["F3889"].astype(str).str.extract(r"(\d+)").astype(float)
    X["F3889_type"] = X["F3889"].astype(str).str[0]

# Drop very high missing columns
missing_rate = X.isna().mean()
high_missing_cols = missing_rate[missing_rate > 0.95].index.tolist()
X = X.drop(columns=high_missing_cols)

# Missing indicators
missing_rate = X.isna().mean()
missing_cols = missing_rate[(missing_rate > 0.05) & (missing_rate < 0.95)].index.tolist()

if len(missing_cols) > 0:
    missing_indicators = X[missing_cols].isna().astype("int8")
    missing_indicators.columns = [col + "_missing" for col in missing_cols]
    X = pd.concat([X, missing_indicators], axis=1)

X = X.copy()

# Drop constant columns
constant_cols = X.columns[X.nunique(dropna=False) <= 1].tolist()
X = X.drop(columns=constant_cols)

# Detect categorical columns
cat_cols = X.select_dtypes(include=["object"]).columns.tolist()

for col in cat_cols:
    X[col] = X[col].astype("category")

cat_feature_indices = [X.columns.get_loc(col) for col in cat_cols]

print("Final shape:", X.shape)
print("Categorical columns:", cat_cols)

# =========================
# 4. CV Setup
# =========================

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scale_pos_weight = (y == 0).sum() / (y == 1).sum()
print("scale_pos_weight:", scale_pos_weight)

lgb_oof = np.zeros(len(X))
cat_oof = np.zeros(len(X))
ensemble_oof = np.zeros(len(X))

lgb_models = []
cat_models = []

# =========================
# 5. LightGBM Params
# =========================

lgb_params = {
    "objective": "binary",
    "metric": ["auc", "binary_logloss"],
    "boosting_type": "gbdt",
    "learning_rate": 0.015,
    "num_leaves": 24,
    "max_depth": -1,
    "min_data_in_leaf": 15,
    "feature_fraction": 0.80,
    "bagging_fraction": 0.80,
    "bagging_freq": 5,
    "lambda_l1": 1.5,
    "lambda_l2": 6.0,
    "scale_pos_weight": scale_pos_weight,
    "verbose": -1,
    "seed": 42
}

# =========================
# 6. Train LightGBM + CatBoost
# =========================

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    print(f"\n========== Fold {fold} ==========")

    X_train = X.iloc[train_idx].copy()
    X_val = X.iloc[val_idx].copy()
    y_train = y.iloc[train_idx]
    y_val = y.iloc[val_idx]

    # ---------- LightGBM ----------
    train_data = lgb.Dataset(
        X_train,
        label=y_train,
        categorical_feature=cat_cols,
        free_raw_data=False
    )

    val_data = lgb.Dataset(
        X_val,
        label=y_val,
        categorical_feature=cat_cols,
        free_raw_data=False
    )

    lgb_model = lgb.train(
        lgb_params,
        train_data,
        valid_sets=[val_data],
        valid_names=["valid"],
        num_boost_round=4000,
        callbacks=[
            lgb.early_stopping(200),
            lgb.log_evaluation(200)
        ]
    )

    lgb_pred = lgb_model.predict(
        X_val,
        num_iteration=lgb_model.best_iteration
    )

    lgb_oof[val_idx] = lgb_pred
    lgb_models.append(lgb_model)

    print("LightGBM ROC-AUC:", roc_auc_score(y_val, lgb_pred))
    print("LightGBM PR-AUC :", average_precision_score(y_val, lgb_pred))

    # ---------- CatBoost ----------
    X_train_cat = X_train.copy()
    X_val_cat = X_val.copy()

    for col in cat_cols:
        X_train_cat[col] = X_train_cat[col].astype(str).fillna("missing")
        X_val_cat[col] = X_val_cat[col].astype(str).fillna("missing")

    cat_model = CatBoostClassifier(
        iterations=3000,
        learning_rate=0.02,
        depth=6,
        loss_function="Logloss",
        eval_metric="AUC",
        class_weights=[1, scale_pos_weight],
        random_seed=42 + fold,
        verbose=200,
        early_stopping_rounds=200,
        l2_leaf_reg=8
    )

    cat_model.fit(
        X_train_cat,
        y_train,
        eval_set=(X_val_cat, y_val),
        cat_features=cat_feature_indices,
        use_best_model=True
    )

    cat_pred = cat_model.predict_proba(X_val_cat)[:, 1]

    cat_oof[val_idx] = cat_pred
    cat_models.append(cat_model)

    print("CatBoost ROC-AUC:", roc_auc_score(y_val, cat_pred))
    print("CatBoost PR-AUC :", average_precision_score(y_val, cat_pred))

# =========================
# 7. Ensemble
# =========================

# Try multiple weights and pick best PR-AUC
best_weight = 0.5
best_pr_auc = 0

for w in np.arange(0.0, 1.01, 0.05):
    temp_pred = (w * lgb_oof) + ((1 - w) * cat_oof)
    score = average_precision_score(y, temp_pred)

    if score > best_pr_auc:
        best_pr_auc = score
        best_weight = w

ensemble_oof = (best_weight * lgb_oof) + ((1 - best_weight) * cat_oof)

print("\n========== MODEL COMPARISON ==========")
print("LightGBM ROC-AUC:", roc_auc_score(y, lgb_oof))
print("LightGBM PR-AUC :", average_precision_score(y, lgb_oof))

print("CatBoost ROC-AUC:", roc_auc_score(y, cat_oof))
print("CatBoost PR-AUC :", average_precision_score(y, cat_oof))

print("Best Ensemble Weight for LightGBM:", best_weight)
print("Ensemble ROC-AUC:", roc_auc_score(y, ensemble_oof))
print("Ensemble PR-AUC :", average_precision_score(y, ensemble_oof))

# =========================
# 8. Threshold Optimization
# =========================

best_f1 = 0
best_threshold = 0.5

for t in np.arange(0.01, 0.99, 0.01):
    pred_label = (ensemble_oof >= t).astype(int)
    score = f1_score(y, pred_label)

    if score > best_f1:
        best_f1 = score
        best_threshold = t

print("\nBest Threshold:", best_threshold)
print("Best F1:", best_f1)

final_pred = (ensemble_oof >= best_threshold).astype(int)

print("\nClassification Report:")
print(classification_report(y, final_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y, final_pred))

# =========================
# 9. Risk Score Output
# =========================

risk_output = pd.DataFrame({
    "actual_label": y.values,
    "lgb_probability": lgb_oof,
    "cat_probability": cat_oof,
    "ensemble_probability": ensemble_oof,
    "risk_score_0_100": (ensemble_oof * 100).round(2),
    "predicted_label": final_pred
})

risk_output.to_csv("optimized_mule_account_risk_scores.csv", index=False)

print("\nSaved: optimized_mule_account_risk_scores.csv")

# =========================
# 10. Feature Importance
# =========================

lgb_importance = pd.DataFrame({
    "feature": X.columns,
    "importance": np.mean(
        [model.feature_importance(importance_type="gain") for model in lgb_models],
        axis=0
    )
}).sort_values("importance", ascending=False)

lgb_importance.to_csv("optimized_lgb_feature_importance.csv", index=False)

print("Saved: optimized_lgb_feature_importance.csv")

print("\nTop 20 LightGBM features:")
print(lgb_importance.head(20))

Original shape: (9082, 3925)
Target distribution:
F3924
0    9001
1      81
Name: count, dtype: int64
Final shape: (9082, 4696)
Categorical columns: ['F3886', 'F3889', 'F3890', 'F3891', 'F3892', 'F3893', 'F3889_type']
scale_pos_weight: 111.12345679012346

========== Fold 1 ==========
Training until validation scores don't improve for 200 rounds
[200]	valid's auc: 1	valid's binary_logloss: 0.00730619
Early stopping, best iteration is:
[1]	valid's auc: 1	valid's binary_logloss: 0.0374427
LightGBM ROC-AUC: 1.0
LightGBM PR-AUC : 1.0
0:	test: 0.9999479	best: 0.9999479 (0)	total: 551ms	remaining: 27m 31s
200:	test: 1.0000000	best: 1.0000000 (1)	total: 1m 21s	remaining: 18m 52s
Stopped by overfitting detector  (200 iterations wait)

bestTest = 1
bestIteration = 1

Shrink model to first 2 iterations.
CatBoost ROC-AUC: 1.0
CatBoost PR-AUC : 1.0

========== Fold 2 ==========
Training until validation scores don't improve for 200 rounds
[200]	valid's auc: 0.998758	valid's binary_logloss: 0.008090